# Machine Learning Application Development - Complete Reboot - by Benajamin AUZANNEAU

## Introduction

In this reboot, you will complete the development of a machine learning application. This includes building and exporting a model pipeline (already provided in a notebook) 📊, loading the pipeline, building a predict function 🧠, and creating an API to serve predictions 🔮. You will then containerize the API using Docker 🐳 and deploy it to the cloud ☁️. A frontend application will be connected to the API to provide a user-friendly interface for making predictions 🖥️.

Follow the instructions carefully, take your time, and have fun deploying! 🎉

## 1. 🚀 Setting Up a New Project on Google Cloud Platform (GCP)

To deploy your app using Google Cloud services, follow these steps to set up a new GCP project:

### 1.1 📋 Creating a New GCP Project

1. **Go to the Google Cloud Console**: [Google Cloud Console](https://console.cloud.google.com/)
2. **Click the project dropdown** in the top navigation bar and select **New Project**.
3. **Fill in the project name**.
4. **Click `Create`** to generate a new project.

### 1.2 📛 Save the Project ID

Once the project is created, you'll see a **project ID**. This ID is important because it will be referenced throughout the rest of the setup. Make sure to save it.

You can find the project ID in the **Google Cloud Console** under **Project Settings** or by selecting your project in the top navigation bar.

In [ ]:
# Store your GCP project information here
GCP_PROJECT_ID = "your-project-id-here"
GCP_REGION = "europe-west1"
ARTIFACT_REPO = "my-artifact-repo"

print(f"Project ID: {GCP_PROJECT_ID}")
print(f"Region: {GCP_REGION}")
print(f"Artifact Repository: {ARTIFACT_REPO}")

### 1.3 🔐 Enable Required APIs

Enable the following APIs for your new project:

1. **Artifact Registry API**: Required for storing and managing Docker images.
2. **Cloud Run API**: Required for deploying your containerized app to Google Cloud Run.

You can enable these by going to the **API & Services** section of the Google Cloud Console and searching for each API.

In [ ]:
# GCloud authentication commands (run in terminal)
commands = [
    "gcloud auth login",
    f"gcloud config set project {GCP_PROJECT_ID}"
]

for cmd in commands:
    print(f"Run in terminal: {cmd}")

## 2. 🏗️ Setting Up a Virtual Environment Using pyenv

### 2.1 🎓 Introduction to pyenv

pyenv is a popular tool used to manage multiple versions of Python on your system. It allows you to easily switch between different versions of Python and create virtual environments tied to specific versions.

### 2.2 🛠️ Installation and Setup

In [ ]:
# Pyenv setup commands (run in terminal)
pyenv_commands = [
    "pyenv install 3.10.6",
    "pyenv virtualenv 3.10.6 myenv",
    "pyenv local myenv",
    "pyenv activate myenv"
]

print("Pyenv setup commands:")
for i, cmd in enumerate(pyenv_commands, 1):
    print(f"{i}. {cmd}")

## 3. 🐍 Creating a Python Package

In this section, we'll walk through the process of creating a Python package that contains a utility function for model predictions. This function will load a model stored in a pickle file, take in four feature values, and return a prediction. 🎯

### 3.1 📓 Familiarize yourself with the Notebook

Before starting, open the notebook file in the `notebooks` folder (`my_notebook.ipynb`). Review its contents to understand the structure and any relevant code or data.

In [ ]:
# Let's start by importing necessary libraries for our ML model
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle
import os

print("Libraries imported successfully!")

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

print(f"Dataset shape: {X.shape}")
print(f"Feature names: {iris.feature_names}")
print(f"Target names: {iris.target_names}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Train a Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model accuracy: {accuracy:.4f}")

In [ ]:
# Create models directory and save the model
os.makedirs('models', exist_ok=True)

# Save the trained model
with open('models/best_model.pkl', 'wb') as file:
    pickle.dump(model, file)

print("Model saved successfully to models/best_model.pkl")

### 3.2 ✏️ Creating the `iris.py` File

Start by creating a folder named `package_folder` at the root of your project directory. Inside this folder, create a file called `iris.py`.

In [ ]:
# Create the package folder structure
os.makedirs('package_folder', exist_ok=True)

# Create iris.py content
iris_py_content = '''import pickle

def my_prediction_function(sepal_length, sepal_width, petal_length, petal_width):
    # Load the model from the pickle file
    with open('models/best_model.pkl', 'rb') as file:
        model = pickle.load(file)

    # Use the model to predict the given inputs
    prediction = model.predict([[sepal_length, sepal_width, petal_length, petal_width]])

    return prediction
'''

# Write iris.py file
with open('package_folder/iris.py', 'w') as f:
    f.write(iris_py_content)

print("iris.py file created successfully!")

In [ ]:
# Create __init__.py file to make it a package
with open('package_folder/__init__.py', 'w') as f:
    f.write('')

print("__init__.py file created successfully!")

### 3.4 📝 Create and Update the `requirements.txt` File

The `requirements.txt` file lists all the Python packages your project depends on.

In [ ]:
# Create requirements.txt
requirements_content = '''scikit-learn
pandas
numpy
fastapi
uvicorn
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("requirements.txt file created successfully!")

### 3.5 🛠️ Creating the `setup.py` File

At the root of your project directory, you'll need to create a `setup.py` file.

In [ ]:
# Create setup.py
setup_py_content = '''from setuptools import find_packages, setup

with open("requirements.txt") as f:
    content = f.readlines()
requirements = [x.strip() for x in content if "git+" not in x]

setup(
    name='package_folder',
    version="0.0.1",
    install_requires=requirements,
    packages=find_packages()
)
'''

with open('setup.py', 'w') as f:
    f.write(setup_py_content)

print("setup.py file created successfully!")

## 4. 🌐 Creating an API

### 4.1 🔌 Introduction to APIs

An API (Application Programming Interface) allows different software systems to communicate with each other. In this exercise, you'll create a RESTful API using FastAPI.

In [ ]:
# Create API file content
api_content = '''from fastapi import FastAPI
import pickle

# FastAPI instance
app = FastAPI()

# Root endpoint
@app.get("/")
def root():
    return {'greeting': "hello"}

# Prediction endpoint
@app.get("/predict")
def predict(sepal_length: float, sepal_width: float, petal_length: float, petal_width: float):
    # Load the model
    with open('models/best_model.pkl', 'rb') as file:
        model = pickle.load(file)

    # Use model to predict inputs
    prediction = model.predict([[sepal_length, sepal_width, petal_length, petal_width]])

    # Return prediction
    return {"prediction": float(prediction[0])}
'''

# Write api_file.py
with open('package_folder/api_file.py', 'w') as f:
    f.write(api_content)

print("api_file.py created successfully!")

### 4.3 🧪 Running the API Locally and Testing

To run the API locally, use the following command in your terminal:

```bash
uvicorn package_folder.api_file:app --reload --port 8000
```

Then access the documentation at: `http://127.0.0.1:8000/docs`

## 5. 🐳 Dockerizing Your Application

### 5.1 🎓 Introduction to Docker

Docker is a platform for developing, shipping, and running applications in containers. Containers are lightweight, standalone, executable packages that include everything needed to run an application.

In [ ]:
# Create Dockerfile content
dockerfile_content = '''FROM python:3.8.12-slim

COPY package_folder package_folder
COPY requirements.txt requirements.txt
COPY models models

RUN pip install -r requirements.txt

CMD uvicorn package_folder.api_file:app --host 0.0.0.0 --port $PORT
'''

# Write Dockerfile
with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)

print("Dockerfile created successfully!")

### 5.4 📁 Using Environment Variables, `.env` and `.envrc` Files

In [ ]:
# Create .env file content
env_content = '''IMAGE=my-api-app
GCP_PROJECT=my-gcp-project
GCP_REGION=europe-west1
ARTIFACTSREPO=my-artifact-repo
MEMORY=512Mi
'''

# Write .env file
with open('.env', 'w') as f:
    f.write(env_content)

# Create .envrc file
envrc_content = 'dotenv\n'

with open('.envrc', 'w') as f:
    f.write(envrc_content)

print(".env and .envrc files created successfully!")

### 5.7 📄 Automating with a Makefile

A Makefile allows you to define build steps as simple commands that you can run from the terminal.

In [ ]:
# Create Makefile content
makefile_content = '''build_container_local:
\tdocker build --tag=${IMAGE}:dev .

run_container_local:
\tdocker run -it -e PORT=8000 -p 8000:8000 ${IMAGE}:dev

build_for_production:
\tdocker build -t ${GCP_REGION}-docker.pkg.dev/${GCP_PROJECT}/${ARTIFACTSREPO}/${IMAGE}:prod .

build_for_production_m1:
\tdocker build --platform linux/amd64 -t ${GCP_REGION}-docker.pkg.dev/${GCP_PROJECT}/${ARTIFACTSREPO}/${IMAGE}:prod .

push_image_production:
\tdocker push ${GCP_REGION}-docker.pkg.dev/${GCP_PROJECT}/${ARTIFACTSREPO}/${IMAGE}:prod

deploy_to_cloud_run:
\tgcloud run deploy --image ${GCP_REGION}-docker.pkg.dev/${GCP_PROJECT}/${ARTIFACTSREPO}/${IMAGE}:prod --memory ${MEMORY} --region ${GCP_REGION}
'''

# Write Makefile
with open('Makefile', 'w') as f:
    f.write(makefile_content)

print("Makefile created successfully!")

## 6. 🌐 Building a Frontend with Streamlit

In this section, you'll create a simple frontend for your application using Streamlit, a fast and easy way to build web applications in Python.

In [ ]:
# Create frontend file content
frontend_content = '''import streamlit as st
import requests

st.title("My awesome MVP")

st.write("Iris predictor")

value1 = st.slider('Select a value for Sepal length', min_value=0.0, max_value=8.0, value=5.0, step=0.1)
value2 = st.slider('Select a value for Sepal width',  min_value=0.0, max_value=5.0, value=3.0, step=0.1)
value3 = st.slider('Select a value for Petal length',  min_value=0.0, max_value=7.0, value=4.0, step=0.1)
value4 = st.slider('Select a value for Petal width',  min_value=0.0, max_value=3.0, value=1.0, step=0.1)

# Replace with your actual deployed API URL
API_URL = "https://your-api-url-here.run.app"

if API_URL != "https://your-api-url-here.run.app":
    try:
        response = requests.get(f"{API_URL}/predict?sepal_length={value1}&sepal_width={value2}&petal_length={value3}&petal_width={value4}")
        if response.status_code == 200:
            prediction = response.json()["prediction"]
            species_names = ["Setosa", "Versicolor", "Virginica"]
            st.write(f"This flower belongs to category: **{species_names[int(prediction)]}**")
        else:
            st.error("Error calling API")
    except Exception as e:
        st.error(f"Error: {e}")
else:
    st.warning("Please update the API_URL with your deployed API endpoint")
'''

print("Frontend content ready to be saved as frontend_file.py")
print("\nFrontend code:")
print(frontend_content)

## 📋 Summary and Next Steps

Congratulations! You have successfully completed the machine learning application development reboot. Here's what you've accomplished:

1. ✅ Set up a Google Cloud Platform project
2. ✅ Created a Python virtual environment with pyenv
3. ✅ Built a Python package with ML prediction functionality
4. ✅ Developed a FastAPI REST API
5. ✅ Containerized the application with Docker
6. ✅ Created deployment automation with Makefile
7. ✅ Built a Streamlit frontend interface

### Next Steps:

1. **Test your API locally** using `uvicorn package_folder.api_file:app --reload --port 8000`
2. **Build and run your Docker container locally** using the Makefile commands
3. **Deploy to Google Cloud Run** using the provided commands
4. **Create and deploy your Streamlit frontend** to connect with your API
5. **Test the complete application end-to-end**

### Terminal Commands Summary:

```bash
# Install package locally
pip install -e .

# Run API locally
uvicorn package_folder.api_file:app --reload --port 8000

# Docker commands
make build_container_local
make run_container_local
make build_for_production
make push_image_production
make deploy_to_cloud_run

# Run Streamlit frontend
streamlit run frontend_file.py
```

Remember to update your `.env` file with your actual GCP project details before deployment! 🚀